In [1]:
import traceback

import torch

torch.manual_seed(0)
torch.set_default_device("cuda")
torch.set_default_dtype(torch.double)
torch.sparse.check_sparse_tensor_invariants.enable()

In [2]:
vals = torch.rand((4,5,3))
vals

tensor([[[0.8805, 0.9397, 0.7129],
         [0.0678, 0.5590, 0.6647],
         [0.5778, 0.5873, 0.8924],
         [0.6570, 0.8501, 0.2644],
         [0.0375, 0.8884, 0.3703]],

        [[0.3868, 0.5426, 0.3016],
         [0.2504, 0.2066, 0.6501],
         [0.6492, 0.7401, 0.7379],
         [0.0063, 0.9987, 0.2476],
         [0.4119, 0.8889, 0.7218]],

        [[0.3503, 0.9443, 0.8562],
         [0.8341, 0.5604, 0.9424],
         [0.6852, 0.9754, 0.4375],
         [0.1688, 0.4968, 0.4205],
         [0.3132, 0.3380, 0.8897]],

        [[0.0710, 0.7271, 0.9482],
         [0.9649, 0.4418, 0.4794],
         [0.5237, 0.3569, 0.6979],
         [0.4915, 0.0489, 0.5817],
         [0.9007, 0.8020, 0.4181]]], device='cuda:0')

In [3]:
l = torch.rand((3,), requires_grad=True)
l

tensor([0.3621, 0.8507, 0.1352], device='cuda:0', requires_grad=True)

In [4]:
dist = torch.cdist(vals / l, vals / l)
dist

tensor([[[0.0000, 2.3164, 1.6227, 3.3773, 3.4423],
         [2.3164, 0.0000, 2.1958, 3.3971, 2.2140],
         [1.6227, 2.1958, 0.0000, 4.6618, 4.1558],
         [3.3773, 3.3971, 4.6618, 0.0000, 1.8827],
         [3.4423, 2.2140, 4.1558, 1.8827, 0.0000]],

        [[0.0000, 2.6354, 3.3169, 1.2457, 3.1360],
         [2.6354, 0.0000, 1.4246, 3.1920, 1.0601],
         [3.3169, 1.4246, 0.0000, 4.0506, 0.6889],
         [1.2457, 3.1920, 4.0506, 0.0000, 3.6850],
         [3.1360, 1.0601, 0.6889, 3.6850, 0.0000]],

        [[0.0000, 1.5477, 3.2334, 3.3048, 0.7614],
         [1.5477, 0.0000, 3.7897, 4.2769, 1.5133],
         [3.2334, 3.7897, 0.0000, 1.5382, 3.5789],
         [3.3048, 4.2769, 1.5382, 0.0000, 3.4991],
         [0.7614, 1.5133, 3.5789, 3.4991, 0.0000]],

        [[0.0000, 4.2706, 2.2760, 3.0557, 4.5431],
         [4.2706, 0.0000, 2.0273, 1.5798, 0.6451],
         [2.2760, 2.0273, 0.0000, 0.9376, 2.3758],
         [3.0557, 1.5798, 0.9376, 0.0000, 1.8776],
         [4.5431, 0.6451,

In [5]:
mask = torch.tril(dist < 1.0, -1)
torch.autograd.grad(dist[mask].sum(), l, retain_graph=True)[0]

tensor([-1.9184, -1.3274, -8.9457], device='cuda:0')

In [6]:
coo = torch.sparse_coo_tensor(mask.nonzero().T, dist[mask], dist.shape).coalesce()
display(coo, torch.autograd.grad(coo.sum(), l, retain_graph=True)[0])

tensor(indices=tensor([[1, 2, 3, 3],
                       [4, 4, 3, 4],
                       [2, 0, 2, 1]]),
       values=tensor([0.6889, 0.7614, 0.9376, 0.6451]),
       device='cuda:0', size=(4, 5, 5), nnz=4, layout=torch.sparse_coo,
       grad_fn=<CoalesceBackward0>)

tensor([-1.9184, -1.3274, -8.9457], device='cuda:0')

In [7]:
coo2 = torch.sparse_coo_tensor(coo.indices(), coo.values() * 2, coo.size())
display(coo2, torch.autograd.grad(coo2.sum(), l, retain_graph=True)[0])

tensor(indices=tensor([[1, 2, 3, 3],
                       [4, 4, 3, 4],
                       [2, 0, 2, 1]]),
       values=tensor([1.3777, 1.5228, 1.8752, 1.2902]),
       device='cuda:0', size=(4, 5, 5), nnz=4, layout=torch.sparse_coo,
       grad_fn=<SparseCooTensorWithDimsAndTensorsBackward0>)

tensor([ -3.8368,  -2.6548, -17.8914], device='cuda:0')

In [8]:
csr = coo[3].coalesce().to_sparse_csr()
display(csr, torch.autograd.grad(dist[3][mask[3]].sum(), l, retain_graph=True)[0], csr, csr.sum())
try:
	torch.autograd.grad(csr.sum(), l, retain_graph=True)[0]
except NotImplementedError as e:
	traceback.print_exception(e)

/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py:122: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  return func(*args, **kwargs)


tensor(crow_indices=tensor([0, 0, 0, 0, 1, 2]),
       col_indices=tensor([2, 1]),
       values=tensor([0.9376, 0.6451]), device='cuda:0', size=(5, 5), nnz=2,
       layout=torch.sparse_csr, grad_fn=<ToSparseCsrBackward0>)

tensor([-0.1580, -0.4908, -8.1973], device='cuda:0')

tensor(crow_indices=tensor([0, 0, 0, 0, 1, 2]),
       col_indices=tensor([2, 1]),
       values=tensor([0.9376, 0.6451]), device='cuda:0', size=(5, 5), nnz=2,
       layout=torch.sparse_csr, grad_fn=<ToSparseCsrBackward0>)

tensor(1.5827, device='cuda:0', grad_fn=<SumBackward0>)

Traceback (most recent call last):
  File "/tmp/ipykernel_33505/2256493162.py", line 4, in <module>
    torch.autograd.grad(csr.sum(), l, retain_graph=True)[0]
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/autograd/__init__.py", line 479, in grad
    return handle_torch_function(
        grad,
    ...<9 lines>...
        materialize_grads=materialize_grads,
    )
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/overrides.py", line 1774, in handle_torch_function
    result = mode.__torch_function__(public_api, types, args, kwargs)
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py", line 122, in __torch_function__
    return func(*args, **kwargs)
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/autograd/__init__.py", line 530, in grad
    result = _engine_run_backward(
        outputs,
    ...<5 lines>...
        accumulate_grad=False,
    )
  Fi

In [9]:
csc = coo[3].coalesce().to_sparse_csc()
display(csc, torch.autograd.grad(dist[3][mask[3]].sum(), l, retain_graph=True)[0], csc, csc.sum())
try:
	torch.autograd.grad(csc.sum(), l, retain_graph=True)[0]
except NotImplementedError as e:
	traceback.print_exception(e)

tensor(ccol_indices=tensor([0, 0, 1, 2, 2, 2]),
       row_indices=tensor([4, 3]),
       values=tensor([0.6451, 0.9376]), device='cuda:0', size=(5, 5), nnz=2,
       layout=torch.sparse_csc, grad_fn=<ToSparseCscBackward0>)

tensor([-0.1580, -0.4908, -8.1973], device='cuda:0')

tensor(ccol_indices=tensor([0, 0, 1, 2, 2, 2]),
       row_indices=tensor([4, 3]),
       values=tensor([0.6451, 0.9376]), device='cuda:0', size=(5, 5), nnz=2,
       layout=torch.sparse_csc, grad_fn=<ToSparseCscBackward0>)

tensor(1.5827, device='cuda:0', grad_fn=<SumBackward0>)

Traceback (most recent call last):
  File "/tmp/ipykernel_33505/2318682950.py", line 4, in <module>
    torch.autograd.grad(csc.sum(), l, retain_graph=True)[0]
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/autograd/__init__.py", line 479, in grad
    return handle_torch_function(
        grad,
    ...<9 lines>...
        materialize_grads=materialize_grads,
    )
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/overrides.py", line 1774, in handle_torch_function
    result = mode.__torch_function__(public_api, types, args, kwargs)
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py", line 122, in __torch_function__
    return func(*args, **kwargs)
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/autograd/__init__.py", line 530, in grad
    result = _engine_run_backward(
        outputs,
    ...<5 lines>...
        accumulate_grad=False,
    )
  Fi

In [10]:
# do it again in cpu
torch.set_default_device("cpu")
vals = vals.cpu()
l = l.cpu().detach().requires_grad_()
dist = torch.cdist(vals / l, vals / l)
mask = dist < 1.0
print(torch.autograd.grad(dist[mask].sum(), l, retain_graph=True)[0])
coo = torch.sparse_coo_tensor(mask.nonzero().T, dist[mask], dist.shape).coalesce()
print(torch.autograd.grad(coo.sum(), l, retain_graph=True)[0])
coo2 = torch.sparse_coo_tensor(coo.indices(), coo.values() * 2, coo.size())
print(torch.autograd.grad(coo2.sum(), l, retain_graph=True)[0])
csr = coo[3].coalesce().to_sparse_csr()
display(torch.autograd.grad(dist[3][mask[3]].sum(), l, retain_graph=True)[0], csr.sum())
try:
	torch.autograd.grad(csr.sum(), l, retain_graph=True)[0]
except NotImplementedError as e:
	traceback.print_exception(e)
csc = coo[3].coalesce().to_sparse_csc()
display(torch.autograd.grad(dist[3][mask[3]].sum(), l, retain_graph=True)[0], csc.sum())
try:
	torch.autograd.grad(csc.sum(), l, retain_graph=True)[0]
except NotImplementedError as e:
	traceback.print_exception(e)

tensor([ -3.8368,  -2.6548, -17.8914])
tensor([ -3.8368,  -2.6548, -17.8914])
tensor([ -7.6735,  -5.3096, -35.7828])


tensor([ -0.3160,  -0.9817, -16.3945])

tensor(3.1653, grad_fn=<SumBackward0>)

Traceback (most recent call last):
  File "/tmp/ipykernel_33505/4019124150.py", line 15, in <module>
    torch.autograd.grad(csr.sum(), l, retain_graph=True)[0]
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/autograd/__init__.py", line 479, in grad
    return handle_torch_function(
        grad,
    ...<9 lines>...
        materialize_grads=materialize_grads,
    )
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/overrides.py", line 1774, in handle_torch_function
    result = mode.__torch_function__(public_api, types, args, kwargs)
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py", line 122, in __torch_function__
    return func(*args, **kwargs)
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/autograd/__init__.py", line 530, in grad
    result = _engine_run_backward(
        outputs,
    ...<5 lines>...
        accumulate_grad=False,
    )
  F

tensor([ -0.3160,  -0.9817, -16.3945])

tensor(3.1653, grad_fn=<SumBackward0>)

Traceback (most recent call last):
  File "/tmp/ipykernel_33505/4019124150.py", line 21, in <module>
    torch.autograd.grad(csc.sum(), l, retain_graph=True)[0]
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/autograd/__init__.py", line 479, in grad
    return handle_torch_function(
        grad,
    ...<9 lines>...
        materialize_grads=materialize_grads,
    )
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/overrides.py", line 1774, in handle_torch_function
    result = mode.__torch_function__(public_api, types, args, kwargs)
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py", line 122, in __torch_function__
    return func(*args, **kwargs)
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/autograd/__init__.py", line 530, in grad
    result = _engine_run_backward(
        outputs,
    ...<5 lines>...
        accumulate_grad=False,
    )
  F

In [11]:
vec = torch.rand((5,))
display(vec)
try:
	display(vec @ coo)
except RuntimeError as e:
	traceback.print_exception(e)
try:
	display(vec.reshape(-1, 1) @ coo)
except RuntimeError as e:
	traceback.print_exception(e)
display(vec @ coo[0])
display(coo[0] @ vec.reshape(-1, 1))
# display(vec @ coo @ vec)

tensor([0.9701, 0.7078, 0.4594, 0.9207, 0.6450])

Traceback (most recent call last):
  File "/tmp/ipykernel_33505/3219486630.py", line 4, in <module>
    display(vec @ coo)
            ~~~~^~~~~
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py", line 122, in __torch_function__
    return func(*args, **kwargs)
RuntimeError: reshape is not implemented for sparse tensors
Traceback (most recent call last):
  File "/tmp/ipykernel_33505/3219486630.py", line 8, in <module>
    display(vec.reshape(-1, 1) @ coo)
            ~~~~~~~~~~~~~~~~~~~^~~~~
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py", line 122, in __torch_function__
    return func(*args, **kwargs)
RuntimeError: expand is unsupported for Sparse tensors


tensor([0., 0., 0., 0., 0.], grad_fn=<SqueezeBackward4>)

tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.]], grad_fn=<MmBackward0>)

In [12]:
coo_l = torch.tril(coo.to_dense(), -1).to_sparse_coo()
display(coo_l)
try:
	display(coo_l + torch.sparse_coo_tensor(torch.arange(coo.shape[-1]).reshape(1, -1).repeat(2, 1), torch.ones(coo.shape[-1]), coo_l.shape[-2:]))
except RuntimeError as e:
	traceback.print_exception(e)

coo_l0 = torch.tril(coo[3].to_dense(), -1).to_sparse_coo()
display(coo_l0 + coo_l0.T + torch.sparse_coo_tensor(torch.arange(coo.shape[-1]).reshape(1, -1).repeat(2, 1), torch.ones(coo.shape[-1]), coo_l0.shape))

tensor(indices=tensor([[1, 2, 3, 3],
                       [4, 4, 3, 4],
                       [2, 0, 2, 1]]),
       values=tensor([0.6889, 0.7614, 0.9376, 0.6451]),
       size=(4, 5, 5), nnz=4, layout=torch.sparse_coo,
       grad_fn=<ToSparseBackward1>)

Traceback (most recent call last):
  File "/tmp/ipykernel_33505/449398016.py", line 4, in <module>
    display(coo_l + torch.sparse_coo_tensor(torch.arange(coo.shape[-1]).reshape(1, -1).repeat(2, 1), torch.ones(coo.shape[-1]), coo_l.shape[-2:]))
            ~~~~~~^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py", line 122, in __torch_function__
    return func(*args, **kwargs)
RuntimeError: add: expected sizes of 'self' and 'other' to match, but [4, 5, 5] != [5, 5]


tensor(indices=tensor([[0, 1, 2, 2, 1, 3, 3, 4, 4],
                       [0, 1, 2, 3, 4, 2, 3, 1, 4]]),
       values=tensor([1.0000, 1.0000, 1.0000, 0.9376, 0.6451, 0.9376, 1.0000,
                      0.6451, 1.0000]),
       size=(5, 5), nnz=9, layout=torch.sparse_coo, grad_fn=<AddBackward0>)

In [13]:
try:
	torch.linalg.eigh(coo)
except NotImplementedError as e:
	traceback.print_exception(e)

try:
	torch.linalg.eigh(coo.to("cuda"))
except NotImplementedError as e:
	traceback.print_exception(e)

try:
	torch.linalg.eigh(coo[-1].to_sparse_csr())
except NotImplementedError as e:
	traceback.print_exception(e)

try:
	torch.linalg.eigh(coo[-1].to_sparse_csr().to("cuda"))
except NotImplementedError as e:
	traceback.print_exception(e)

Traceback (most recent call last):
  File "/tmp/ipykernel_33505/922684469.py", line 2, in <module>
    torch.linalg.eigh(coo)
    ~~~~~~~~~~~~~~~~~^^^^^
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py", line 122, in __torch_function__
    return func(*args, **kwargs)
NotImplementedError: Could not run 'aten::_linalg_eigh' with arguments from the 'SparseCPU' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'aten::_linalg_eigh' is only available for these backends: [CPU, CUDA, HIP, MPS, IPU, XPU, HPU, VE, MTIA, MAIA, PrivateUse1, PrivateUse2, PrivateUse3, Meta, FPGA, Vulkan, Metal, QuantizedCPU, QuantizedCUDA, QuantizedHIP, QuantizedMPS, QuantizedIPU, QuantizedXPU, QuantizedHPU, QuantizedVE, QuantizedMTIA, QuantizedMAIA, Quant

In [14]:
def wendland(r: torch.Tensor) -> torch.Tensor:
	return (1 - r) ** 4 * (4 * r + 1)

N_PTS = 100
pts = torch.rand(N_PTS, 3)
dist = torch.cdist(pts / l, pts / l).to("cuda")
mask = dist < 1
coo_large = torch.sparse_coo_tensor(mask.nonzero().T, wendland(dist[mask]), dist.shape).coalesce()
laplacian = torch.sparse_coo_tensor(torch.arange(N_PTS).reshape(1, -1).repeat(2, 1), coo_large.sum(-1).to_dense(), coo_large.shape) - coo_large
display(torch.lobpcg(laplacian, k=10, largest=False, niter=-1)[0], torch.linalg.eigvalsh(laplacian.to_dense())[:10])

pts2 = torch.rand(N_PTS, 3)
dist2 = torch.cdist(pts2 / l, pts2 / l).to("cuda")
mask2 = dist2 < 1
mask2[N_PTS//2:,:N_PTS//2] = False
mask2[:N_PTS//2,N_PTS//2:] = False
coo_large2 = torch.sparse_coo_tensor(mask2.nonzero().T, wendland(dist2[mask2]), dist2.shape).coalesce()
laplacian2 = torch.sparse_coo_tensor(torch.arange(N_PTS).reshape(1, -1).repeat(2, 1), coo_large2.sum(-1).to_dense(), coo_large2.shape) - coo_large2
display(torch.lobpcg(laplacian2, k=20, largest=False, niter=-1)[0], torch.linalg.eigvalsh(laplacian2.to_dense())[:20])

tensor([8.3069e-16, 6.0499e-05, 2.4177e-03, 5.5187e-03, 1.5976e-02, 1.6192e-02,
        2.0415e-02, 2.1587e-02, 2.6409e-02, 3.6636e-02],
       grad_fn=<LOBPCGAutogradFunctionBackward>)

tensor([-6.0014e-16,  6.0499e-05,  2.4177e-03,  5.5187e-03,  1.5976e-02,
         1.6192e-02,  2.0415e-02,  2.1587e-02,  2.6409e-02,  3.6636e-02],
       grad_fn=<SliceBackward0>)

tensor([1.4036e-16, 3.0340e-16, 2.0582e-16, 1.5995e-16, 2.5688e-16, 1.5508e-16,
        1.9607e-16, 3.1890e-16, 6.5416e-16, 1.5739e-08, 1.5050e-06, 6.4583e-06,
        4.7562e-04, 1.1647e-03, 1.5633e-03, 4.8508e-03, 5.6047e-03, 6.4744e-03,
        6.8073e-03, 7.1889e-03], grad_fn=<LOBPCGAutogradFunctionBackward>)

tensor([-5.4021e-16, -2.6029e-16, -1.5779e-16, -7.5973e-17, -6.8786e-17,
         0.0000e+00,  3.6814e-17,  1.0174e-16,  1.0367e-16,  1.5739e-08,
         1.5050e-06,  6.4583e-06,  4.7562e-04,  1.1647e-03,  1.5633e-03,
         4.8508e-03,  5.6047e-03,  6.4744e-03,  6.8073e-03,  7.1889e-03],
       grad_fn=<SliceBackward0>)

In [23]:
import collections.abc

import torchsparsegradutils as tsgu
from torchsparsegradutils.utils.minres import MINRESSettings, minres as minres_lib
from torchsparsegradutils.utils.bicgstab import bicgstab


def _pad_with_singletons(obj, num_singletons_before=0, num_singletons_after=0):
    """
    Pad obj with singleton dimensions on the left and right
    Example:
        >>> x = torch.randn(10, 5)
        >>> _pad_with_singletons(x, 2, 3).shape
        torch.Size([1, 1, 10, 5, 1, 1, 1])
    """
    new_shape = [1] * num_singletons_before + list(obj.shape) + [1] * num_singletons_after
    return obj.view(*new_shape)


def minres(
    matmul_closure: torch.Tensor | collections.abc.Callable[[torch.Tensor], torch.Tensor],
    rhs: torch.Tensor,
    eps: float = 1e-25,
    shifts: torch.Tensor | None = None,
    value: float | None = None,
    max_iter: int | None = None,
    preconditioner: collections.abc.Callable[[torch.Tensor], torch.Tensor] | None = None,
    settings: MINRESSettings = MINRESSettings(),
) -> torch.Tensor:
    """
    Minimum Residual (MINRES) solver for symmetric (Hermitian) linear systems.

    Solves linear systems ``A x = b`` where ``A`` is symmetric (Hermitian) and may
    be indefinite. Supports single/multiple right-hand sides and (optionally)
    multiple shift values to solve ``(A + \\sigma I) x = b`` in one run. Gradually
    minimizes the residual norm ``||A x - b||_2`` via the Lanczos process.

    Parameters
    ----------
    matmul_closure : {torch.Tensor, callable(x) -> A @ x}
        Matrix–vector multiplication operator. If a tensor is provided, its
        ``.matmul`` is used. The operator should represent a symmetric/Hermitian
        matrix for MINRES to behave as intended.
    rhs : torch.Tensor, shape (..., n) or (..., n, k)
        Right-hand side vector(s). Leading batch dimensions are supported; for
        multi-RHS, the last two dims are ``(n, k)``.
    eps : float, optional
        Small constant to prevent division by zero/numerical issues. Default: 1e-25.
    shifts : torch.Tensor or scalar, optional
        Shift(s) ``\\sigma`` for solving ``(A + \\sigma I) x = b``. If ``None`` or a
        scalar, a single system is solved. If a tensor with ``s`` elements, the
        solver computes ``s`` shifted systems and stacks their solutions along a
        new leading dimension.
    value : float, optional
        Scalar multiplier ``\\alpha`` applied to the operator (solves ``(\\alpha A) x = b``)
        when provided. Default: ``None`` (no scaling).
    max_iter : int, optional
        Maximum iterations. If ``None``, uses ``settings.max_cg_iterations``.
        Internally capped at ``n + 1`` where ``n`` is the problem size.
    preconditioner : callable, optional
        Left preconditioner with signature ``preconditioner(x) -> M^{-1} x``.
        If ``None``, no preconditioning is used.
    settings : MINRESSettings, optional
        Configuration object controlling iteration caps and tolerances
        (e.g., ``minres_tolerance`` for the relative update criterion).

    Returns
    -------
    torch.Tensor
        If ``shifts`` is ``None`` or a scalar: solution with the **same shape as**
        ``rhs`` (i.e., ``(..., n)`` or ``(..., n, k)``).
        If ``shifts`` has length ``s``: a stacked tensor of shape
        ``(s, *rhs.shape)`` containing solutions for each shift.

    Raises
    ------
    RuntimeError
        If ``matmul_closure`` is neither a tensor nor a callable.

    Notes
    -----
    - MINRES [1g]_ is appropriate for symmetric/Hermitian **indefinite** systems; it
      minimizes the Euclidean residual norm rather than the A-norm (as in CG).
    - For symmetric positive definite systems, Conjugate Gradient (CG) typically
      converges faster; prefer CG unless indefiniteness/robustness suggests MINRES.
    - When multiple shifts are provided, the solver reuses Lanczos information and
      returns one solution per shift value.
    - All inputs should share device and dtype; the implementation normalizes
      ``rhs`` internally and rescales the final solution(s).

    See Also
    --------
    linear_cg : Conjugate Gradient for SPD systems.
    bicgstab : BiCGSTAB for general non-symmetric systems.

    References
    ----------
    .. [1g] Paige, C. C., & Saunders, M. A. (1975). Solution of sparse indefinite
           systems of linear equations. *SIAM Journal on Numerical Analysis*, 12(4), 617–629.

    Examples
    --------
    Basic solve (indefinite, symmetric):

    >>> A = torch.tensor([[2.0, 1.0], [1.0, -1.0]])
    >>> b = torch.tensor([1.0, 2.0])
    >>> x = minres(A.matmul, b)
    >>> x.shape
    torch.Size([2])

    Multiple right-hand sides:

    >>> B = torch.randn(2, 3)
    >>> X = minres(A.matmul, B)
    >>> X.shape
    torch.Size([2, 3])

    Shifted system (regularization):

    >>> x_shifted = minres(A.matmul, b, shifts=torch.tensor(0.1))

    Sparse operator via closure:

    >>> idx = torch.tensor([[0, 0, 1, 1], [0, 1, 0, 1]])
    >>> val = torch.tensor([2.0, 1.0, 1.0, -1.0])
    >>> A_sp = torch.sparse_coo_tensor(idx, val, (2, 2))
    >>> x = minres(lambda v: A_sp @ v, b)

    With a simple diagonal preconditioner:

    >>> M_diag = torch.abs(torch.diag(A)) + 0.1
    >>> precond = lambda x: x / M_diag.unsqueeze(-1)
    >>> x = minres(A.matmul, b, preconditioner=precond)

    Custom iteration cap/tolerance:

    >>> settings = MINRESSettings(max_cg_iterations=200, minres_tolerance=1e-5)
    >>> x = minres(A.matmul, b, settings=settings)
    """
    # Default values
    if torch.is_tensor(matmul_closure):
        matmul_closure = matmul_closure.matmul
    mm_ = matmul_closure
    if preconditioner is None:
        preconditioner = lambda x: x.clone()

    if shifts is None:
        shifts = torch.tensor(0.0, dtype=rhs.dtype, device=rhs.device)

    # Scale the rhs
    squeeze = False
    if rhs.dim() == 1:
        rhs = rhs.unsqueeze(-1)
        squeeze = True

    rhs_norm = rhs.norm(2, dim=-2, keepdim=True)
    rhs_is_zero = rhs_norm.lt(1e-10)
    rhs_norm = rhs_norm.masked_fill_(rhs_is_zero, 1)
    rhs = rhs.div(rhs_norm)

    # Use the right number of iterations
    if max_iter is None:
        max_iter = settings.max_cg_iterations
    max_iter = min(max_iter, rhs.size(-2) + 1)

    # Epsilon (to prevent nans)
    eps = torch.tensor(eps, dtype=rhs.dtype, device=rhs.device) # pyright: ignore[reportAssignmentType]

    # Create space for matmul product, solution
    prod = mm_(rhs)
    if value is not None:
        prod.mul_(value)

    # Resize shifts
    shifts = _pad_with_singletons(shifts, 0, prod.dim() - shifts.dim() + 1)
    solution = torch.zeros(shifts.shape[:1] + prod.shape, dtype=rhs.dtype, device=rhs.device)

    # Variables for Lanczos terms
    zvec_prev2 = torch.zeros_like(prod)
    zvec_prev1 = rhs.clone().expand_as(prod).contiguous()
    qvec_prev1 = preconditioner(zvec_prev1)
    alpha_curr = torch.empty(prod.shape[:-2] + (1, prod.size(-1)), dtype=rhs.dtype, device=rhs.device)
    alpha_shifted_curr = torch.empty(solution.shape[:-2] + (1, prod.size(-1)), dtype=rhs.dtype, device=rhs.device)
    beta_prev = (zvec_prev1 * qvec_prev1).sum(dim=-2, keepdim=True).sqrt_()
    beta_curr = torch.empty_like(beta_prev)
    tmpvec = torch.empty_like(qvec_prev1)

    # Divide by beta_prev
    zvec_prev1.div_(beta_prev)
    qvec_prev1.div_(beta_prev)

    # Variables for the QR rotation
    # 1) Components of the Givens rotations
    cos_prev2 = torch.ones(solution.shape[:-2] + (1, rhs.size(-1)), dtype=rhs.dtype, device=rhs.device)
    sin_prev2 = torch.zeros(solution.shape[:-2] + (1, rhs.size(-1)), dtype=rhs.dtype, device=rhs.device)
    cos_prev1 = torch.ones_like(cos_prev2)
    sin_prev1 = torch.zeros_like(sin_prev2)
    radius_curr = torch.empty_like(cos_prev1)
    cos_curr = torch.empty_like(cos_prev1)
    sin_curr = torch.empty_like(cos_prev1)
    # 2) Terms QR decomposition of T
    subsub_diag_term = torch.empty_like(alpha_shifted_curr)
    sub_diag_term = torch.empty_like(alpha_shifted_curr)
    diag_term = torch.empty_like(alpha_shifted_curr)

    # Variables for the solution updates
    # 1) The "search" vectors of the solution
    # Equivalent to the vectors of Q R^{-1}, where Q is the matrix of Lanczos vectors and
    # R is the QR factor of the tridiagonal Lanczos matrix.
    search_prev2 = torch.zeros_like(solution)
    search_prev1 = torch.zeros_like(solution)
    search_curr = torch.empty_like(search_prev1)
    search_update = torch.empty_like(search_prev1)
    # 2) The "scaling" terms of the search vectors
    # Equivalent to the terms of V^T Q^T rhs, where Q is the matrix of Lanczos vectors and
    # V is the QR orthonormal of the tridiagonal Lanczos matrix.
    scale_prev = beta_prev.repeat(shifts.size(0), *([1] * beta_prev.dim()))
    scale_curr = torch.empty_like(scale_prev)

    # Terms for checking for convergence
    solution_norm = torch.zeros(*solution.shape[:-2], solution.size(-1), dtype=solution.dtype, device=solution.device)
    search_update_norm = torch.zeros_like(solution_norm)

    # Maybe log
    if settings.verbose_linalg:
        # settings.verbose_linalg.logger.debug(
        print(
            f"Running MINRES on a {rhs.shape} RHS for {max_iter} iterations (tol={settings.minres_tolerance}). "
            f"Output: {solution.shape}."
        )

    # Perform iterations
    for i in range(max_iter + 2):
        # Perform matmul
        prod = mm_(qvec_prev1)
        if value is not None:
            prod.mul_(value)

        # Get next Lanczos terms
        # --> alpha_curr, beta_curr, qvec_curr
        tmpvec[...] = torch.mul(prod, qvec_prev1)
        alpha_curr[...] = torch.sum(tmpvec, -2, keepdim=True)

        zvec_curr = prod.addcmul_(alpha_curr, zvec_prev1, value=-1).addcmul_(beta_prev, zvec_prev2, value=-1)

        qvec_curr = preconditioner(zvec_curr)
        tmpvec[...] = torch.mul(zvec_curr, qvec_curr)
        beta_curr[...] =torch.sum(tmpvec, -2, keepdim=True)
        beta_curr.sqrt_()
        beta_curr.clamp_min_(eps)

        zvec_curr.div_(beta_curr)
        qvec_curr.div_(beta_curr)

        # Perform JIT-ted update
        conv = _jit_minres_updates(
            solution,
            shifts,
            eps,
            qvec_prev1,
            alpha_curr,
            alpha_shifted_curr,
            beta_prev,
            beta_curr,
            cos_prev2,
            cos_prev1,
            cos_curr,
            sin_prev2,
            sin_prev1,
            sin_curr,
            radius_curr,
            subsub_diag_term,
            sub_diag_term,
            diag_term,
            search_prev2,
            search_prev1,
            search_curr,
            search_update,
            scale_prev,
            scale_curr,
            search_update_norm,
            solution_norm,
        )

        # Check convergence criterion
        if (i + 1) % 10 == 0:
            search_update_norm[...] = torch.norm(search_update, dim=-2)
            solution_norm[...] =torch.norm(solution, dim=-2)
            conv = search_update_norm.div_(solution_norm).mean().item()
            if conv < settings.minres_tolerance:
                break

        # Update terms for next iteration
        # Lanczos terms
        zvec_prev2, zvec_prev1 = zvec_prev1, prod
        qvec_prev1 = qvec_curr
        beta_prev, beta_curr = beta_curr, beta_prev
        # Givens rotations terms
        cos_prev2, cos_prev1, cos_curr = cos_prev1, cos_curr, cos_prev2
        sin_prev2, sin_prev1, sin_curr = sin_prev1, sin_curr, sin_prev2
        # Search vector terms)
        search_prev2, search_prev1, search_curr = search_prev1, search_curr, search_prev2
        scale_prev, scale_curr = scale_curr, scale_prev

    # For rhs-s that are close to zero, set them to zero
    solution.masked_fill_(rhs_is_zero, 0)

    if squeeze:
        solution = solution.squeeze(-1)
        rhs = rhs.squeeze(-1)
        rhs_norm = rhs_norm.squeeze(-1)

    if shifts.numel() == 1:
        # If we weren't shifting we shouldn't return a batch output
        solution = solution.squeeze(0)

    return solution.mul_(rhs_norm)


def _jit_minres_updates(
    solution,
    shifts,
    eps,
    qvec_prev1,
    alpha_curr,
    alpha_shifted_curr,
    beta_prev,
    beta_curr,
    cos_prev2,
    cos_prev1,
    cos_curr,
    sin_prev2,
    sin_prev1,
    sin_curr,
    radius_curr,
    subsub_diag_term,
    sub_diag_term,
    diag_term,
    search_prev2,
    search_prev1,
    search_curr,
    search_update,
    scale_prev,
    scale_curr,
    search_update_norm,
    solution_norm,
):
    # Start givens rotation
    # Givens rotation from 2 steps ago
    subsub_diag_term[...] = torch.mul(sin_prev2, beta_prev)
    sub_diag_term[...] = torch.mul(cos_prev2, beta_prev)

    # Compute shifted alpha
    alpha_shifted_curr[...] = torch.add(alpha_curr, shifts)

    # Givens rotation from 1 step ago
    diag_term[...] =torch.mul(alpha_shifted_curr, cos_prev1)
    diag_term.addcmul_(sin_prev1, sub_diag_term, value=-1)
    sub_diag_term.mul_(cos_prev1).addcmul_(sin_prev1, alpha_shifted_curr)

    # 3) Compute next Givens terms
    radius_curr[...] = torch.mul(diag_term, sub_diag_term)
    radius_curr.addcmul_(beta_curr, beta_curr).sqrt_()
    cos_curr[...] = torch.div(diag_term, radius_curr)
    sin_curr[...] = torch.div(beta_curr, radius_curr)
    # 4) Apply current Givens rotation
    diag_term.mul_(cos_curr).addcmul_(sin_curr, beta_curr)

    # Update the solution
    # --> search_curr, scale_curr solution
    # 1) Apply the latest Givens rotation to the Lanczos-rhs ( ||rhs|| e_1 )
    # This is getting the scale terms for the "search" vectors
    scale_curr[...] = torch.mul(scale_prev, sin_curr)
    scale_curr.mul_(-1)
    scale_prev.mul_(cos_curr)
    # 2) Get the new search vector
    search_curr[...] = torch.addcmul(qvec_prev1, sub_diag_term, search_prev1, value=-1)
    search_curr.addcmul_(subsub_diag_term, search_prev2, value=-1)
    search_curr.div_(diag_term)

    # 3) Update the solution
    search_update[...] = torch.mul(search_curr, scale_prev)
    solution.add_(search_update)

display(torch.autograd.grad(tsgu.sparse_generic_solve(coo_large, pts, minres).sum(), l, retain_graph=True)[0])
try:
    torch.autograd.grad(tsgu.sparse_generic_solve(coo_large, pts, minres_lib).sum(), l, create_graph=True, retain_graph=True)[0]
except RuntimeError as e:
	traceback.print_exception(e)
grad = torch.autograd.grad(tsgu.sparse_generic_solve(coo_large, pts, minres).sum(), l, create_graph=True, retain_graph=True)[0]
display(grad)
display(torch.autograd.grad(grad.sum(), l, retain_graph=True)[0])
grad2 = torch.autograd.grad(tsgu.sparse_generic_solve(coo_large, pts, bicgstab).sum(), l, create_graph=True, retain_graph=True)[0]
display(grad)
display(torch.autograd.grad(grad.sum(), l, retain_graph=True)[0])

tensor([ -83.8884,  -29.5299, -254.1182])

Traceback (most recent call last):
  File "/tmp/ipykernel_33505/3880307540.py", line 387, in <module>
    torch.autograd.grad(tsgu.sparse_generic_solve(coo_large, pts, minres_lib).sum(), l, create_graph=True, retain_graph=True)[0]
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/autograd/__init__.py", line 479, in grad
    return handle_torch_function(
        grad,
    ...<9 lines>...
        materialize_grads=materialize_grads,
    )
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/overrides.py", line 1774, in handle_torch_function
    result = mode.__torch_function__(public_api, types, args, kwargs)
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/utils/_device.py", line 122, in __torch_function__
    return func(*args, **kwargs)
  File "/home/kaigu/.venv/venv/lib/python3.14/site-packages/torch/autograd/__ini

tensor([ -83.8884,  -29.5299, -254.1182], grad_fn=<AddBackward0>)

tensor([ -839.5283,  -365.5247, -2310.2719])

/home/kaigu/.venv/venv/lib/python3.14/site-packages/torchsparsegradutils/utils/bicgstab.py:170: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  settings.logger.info("Initial residual = %8.2e" % residNorm0)


tensor([ -83.8884,  -29.5299, -254.1182], grad_fn=<AddBackward0>)

tensor([ -839.5283,  -365.5247, -2310.2719])